# HW7. Часові ряди та напівструктуровані дані в ML: NYC Taxi і JSONB

Дані: NYC TLC Yellow Taxi (yellow_tripdata_2024-03) для time-series частини та синтетичний event-log для JSONB. СУБД: локальний PostgreSQL 16 (pgAdmin), база hw7.

## Завдання 1. Setup і завантаження даних

In [1]:
%pip install -q psycopg2-binary sqlalchemy pandas pyarrow requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from urllib.request import urlretrieve
import getpass
import pandas as pd
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Пароль не зберігається у файлі: береться зі змінної середовища або вводиться вручну.
DB_URL = os.getenv("HW7_DB_URL")
if not DB_URL:
    DB_URL = f"postgresql+psycopg2://postgres:{getpass.getpass('Пароль користувача postgres: ')}@localhost:5432/hw7"

engine = create_engine(DB_URL, future=True)

with engine.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())


def run(sql_text):
    """Виконує SQL (можна кілька команд). Якщо остання повертає рядки, віддає DataFrame."""
    with engine.begin() as conn:
        # psycopg2 трактує % як параметр, тому подвоюємо (оператор залишку, LIKE тощо)
        res = conn.exec_driver_sql(sql_text.replace("%", "%%"))
        if res.returns_rows:
            return pd.DataFrame(res.fetchall(), columns=list(res.keys()))


def show_plan(sql_text):
    """Друкує output EXPLAIN (ANALYZE, BUFFERS) як текст, щоб він зберігся у notebook."""
    plan = run(sql_text)
    print("\n".join(plan.iloc[:, 0]))

PostgreSQL 16.15, compiled by Visual C++ build 1944, 64-bit


### Варіант A: NYC Taxi з офіційного Parquet

Файл завантажується з офіційного джерела TLC. Якщо завантаження неможливе, notebook автоматично переходить на Варіант B (stub-schema) і явно виводить це повідомлення.

In [3]:
URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2024-03.parquet"
)

parquet_path = os.getenv("TAXI_PARQUET") or os.path.join(os.getcwd(), "yellow_tripdata_2024-03.parquet")

USE_STUB = False
try:
    if not os.path.exists(parquet_path):
        urlretrieve(URL, parquet_path)
    print("Parquet:", parquet_path, os.path.getsize(parquet_path), "bytes")
except Exception as exc:
    USE_STUB = True
    print("Не вдалося отримати Parquet:", repr(exc))
    print("Використовується Варіант B (stub-schema).")

print("USE_STUB =", USE_STUB)

Parquet: c:\Users\PC\Downloads\yellow_tripdata_2024-03.parquet 60078280 bytes
USE_STUB = False


In [4]:
columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "fare_amount",
    "passenger_count",
]

if not USE_STUB:
    df = pd.read_parquet(parquet_path, columns=columns)
    print("рядків у файлі:", len(df))

    # залишаємо лише поїздки березня 2024 (у TLC-файлах бувають рядки з хибними датами)
    df = df[(df["tpep_pickup_datetime"] >= "2024-03-01") & (df["tpep_pickup_datetime"] < "2024-04-01")]

    # basic cleaning before loading to PostgreSQL
    df = df[(df["fare_amount"] > 0) & (df["fare_amount"] < 500)]
    df = df[(df["trip_distance"] > 0) & (df["trip_distance"] < 100)]
    df = df[df["PULocationID"].notna()]

    # випадкова вибірка 500 000 рядків: перші рядки файла здебільшого належать першим дням місяця,
    # а для lag-30d потрібні всі дні, тому береться вибірка, а не head()
    df = df.sample(n=min(500_000, len(df)), random_state=42)

    df.to_sql("nyc_taxi_raw", engine, if_exists="replace", index=False, chunksize=10_000)
    print(pd.read_sql("SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw", engine))

рядків у файлі: 3582628
   n_rows
0  500000


Daily aggregation t7_hw_taxi_daily: з real-даних (Варіант A) або зі stub-схеми (Варіант B). Саме ця таблиця використовується в усіх time-series запитах.

In [5]:
DAILY_SQL = """
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100
  AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
"""

STUB_SQL = """
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily (
    pickup_zone_id  INTEGER NOT NULL,
    ts_day          DATE    NOT NULL,
    trip_count      INTEGER NOT NULL CHECK (trip_count >= 0),
    avg_fare        NUMERIC(10, 2) NOT NULL CHECK (avg_fare >= 0),
    total_distance  NUMERIC(12, 2) NOT NULL CHECK (total_distance >= 0),
    PRIMARY KEY (pickup_zone_id, ts_day)
);

INSERT INTO t7_hw_taxi_daily (
    pickup_zone_id,
    ts_day,
    trip_count,
    avg_fare,
    total_distance
)
SELECT
    z.zone_id,
    DATE '2024-03-31' - (59 - n) AS ts_day,
    500 + (n * 7) + z.zone_id * 50 + (n % 5) * 30 AS trip_count,
    (12.5 + z.zone_id * 0.7 + (n % 4) * 0.4)::NUMERIC(10, 2) AS avg_fare,
    (1500 + n * 25 + z.zone_id * 120)::NUMERIC(12, 2) AS total_distance
FROM generate_series(0, 59) AS n
CROSS JOIN (VALUES (1), (2), (3), (4), (5)) AS z(zone_id);
"""

run(STUB_SQL if USE_STUB else DAILY_SQL)
print("Джерело t7_hw_taxi_daily:", "stub-schema (Варіант B)" if USE_STUB else "NYC Taxi 2024-03 (Варіант A)")

Джерело t7_hw_taxi_daily: NYC Taxi 2024-03 (Варіант A)


In [6]:
run("""
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;
""")

,pickup_zone_id,n_days
0,1,7
1,2,2
2,3,15
3,4,31
4,6,6
5,7,31
6,8,1
7,9,6
8,10,31
9,11,14


In [7]:
# контроль: у таблиці має бути понад 100 рядків
n_daily = run("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT pickup_zone_id) AS n_zones, MIN(ts_day) AS first_day, MAX(ts_day) AS last_day FROM t7_hw_taxi_daily")
print(n_daily)
assert int(n_daily.loc[0, "n_rows"]) > 100

   n_rows  n_zones   first_day    last_day
0    5445      250  2024-03-01  2024-03-31


Production-style збереження моменту часу: окрема typed-таблиця з TIMESTAMPTZ. Raw timestamp TLC є локальним нью-йоркським часом, тому він інтерпретується через AT TIME ZONE 'America/New_York'. Для Варіанта B (stub) сирих поїздок немає, тож цей крок пропускається.

In [8]:
if not USE_STUB:
    run("""
    DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

    CREATE TABLE t7_hw_taxi_trips_typed AS
    SELECT
        "PULocationID"::INTEGER AS pickup_zone_id,
        "DOLocationID"::INTEGER AS dropoff_zone_id,
        (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
        (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
        trip_distance::NUMERIC(10, 2) AS trip_distance,
        fare_amount::NUMERIC(10, 2) AS fare_amount,
        passenger_count::INTEGER AS passenger_count
    FROM nyc_taxi_raw
    WHERE fare_amount > 0
      AND fare_amount < 500
      AND trip_distance > 0
      AND trip_distance < 100;
    """)
    display(run("""
    SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
    FROM t7_hw_taxi_trips_typed
    LIMIT 5;
    """))
else:
    print("Stub-schema: t7_hw_taxi_trips_typed пропущено (немає сирих поїздок).")

,pickup_at,pickup_type
0,2024-03-16 18:52:24+02:00,timestamp with time zone
1,2024-03-10 07:12:10+02:00,timestamp with time zone
2,2024-03-22 17:40:48+02:00,timestamp with time zone
3,2024-03-03 01:05:51+02:00,timestamp with time zone
4,2024-03-27 02:22:26+02:00,timestamp with time zone


## Завдання 2. Time-series ML features

Усі ознаки рахуються на рівні pickup-zone, тобто з PARTITION BY pickup_zone_id.

### Feature 1. Lag features: lag-1d, lag-7d, lag-30d

In [9]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-13,1,NaN,NaN,None
1,1,2024-03-14,1,1.0,NaN,None
2,1,2024-03-15,1,1.0,NaN,None
3,1,2024-03-16,1,1.0,NaN,None
4,1,2024-03-22,1,1.0,NaN,None
5,1,2024-03-23,1,1.0,NaN,None
6,1,2024-03-26,1,1.0,NaN,None
7,2,2024-03-09,1,NaN,NaN,None
8,2,2024-03-20,1,1.0,NaN,None
9,3,2024-03-01,1,NaN,NaN,None


На перших 30 рядках кожної zone trips_lag_30d дорівнює NULL, бо попередньої історії ще немає. Для training dataset із повною історією такі рядки відфільтровуються через WHERE trips_lag_30d IS NOT NULL.

### Feature 2. Rolling 7-day average fare

Класичне «7 календарних днів включно з поточним» задається як INTERVAL '6 days' PRECEDING. ts_day приводиться до TIMESTAMP, щоб frame RANGE ... INTERVAL був однозначним.

In [10]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-13,117.00,117.0000000000000000,1
1,1,2024-03-14,29.60,73.3000000000000000,2
2,1,2024-03-15,98.60,81.7333333333333333,3
3,1,2024-03-16,263.40,127.1500000000000000,4
4,1,2024-03-22,70.00,166.7000000000000000,2
5,1,2024-03-23,54.80,62.4000000000000000,2
6,1,2024-03-26,80.00,68.2666666666666667,3
7,2,2024-03-09,51.30,51.3000000000000000,1
8,2,2024-03-20,19.10,19.1000000000000000,1
9,3,2024-03-01,50.50,50.5000000000000000,1


### Feature 3. Rolling 30-day fare volatility

In [11]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-03-13,117.00,None,1
1,1,2024-03-14,29.60,61.8011326757042536,2
2,1,2024-03-15,98.60,46.0766028840379392,3
3,1,2024-03-16,263.40,98.3161397397870445,4
4,1,2024-03-22,70.00,88.8975365237980034,5
5,1,2024-03-23,54.80,83.3112157315368113,6
6,1,2024-03-26,80.00,76.6638425980902806,7
7,2,2024-03-09,51.30,None,1
8,2,2024-03-20,19.10,22.7688383542068303,2
9,3,2024-03-01,50.50,None,1


### Feature 4. Expanding cumulative trips

In [12]:
run("""
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-13,1,1,1.00000000000000000000
1,1,2024-03-14,1,2,1.00000000000000000000
2,1,2024-03-15,1,3,1.00000000000000000000
3,1,2024-03-16,1,4,1.00000000000000000000
4,1,2024-03-22,1,5,1.00000000000000000000
5,1,2024-03-23,1,6,1.00000000000000000000
6,1,2024-03-26,1,7,1.00000000000000000000
7,2,2024-03-09,1,1,1.00000000000000000000
8,2,2024-03-20,1,2,1.00000000000000000000
9,3,2024-03-01,1,1,1.00000000000000000000


### Combined feature view

Усі ознаки зібрано в одному view. У кожному window definition є PARTITION BY pickup_zone_id.

In [13]:
run("""
CREATE OR REPLACE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
""")

In [14]:
run("""
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
""")

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,4,2024-03-31,57,19.55,188.71,61,58,26,17.6857142857142857,3.1479601313399360,986,31.8064516129032258
1,7,2024-03-31,17,25.36,91.77,13,16,11,25.0328571428571429,7.1856903219035852,302,9.7419354838709677
2,10,2024-03-31,3,31.28,20.63,5,7,7,45.6028571428571429,10.1732105037703308,190,6.1290322580645161
3,13,2024-03-31,49,22.69,223.55,65,53,95,23.4714285714285714,1.7060770690438464,2925,94.3548387096774194
4,24,2024-03-31,35,14.43,85.93,42,32,52,16.4457142857142857,1.5994775008927350,1281,41.3225806451612903
5,25,2024-03-31,12,25.98,59.34,6,9,11,25.4371428571428571,7.4413812214372286,249,8.0322580645161290
6,33,2024-03-31,10,32.00,71.52,9,11,6,24.8528571428571429,4.3321081910901469,313,10.0967741935483871
7,39,2024-03-31,3,18.10,7.65,5,2,5,32.4642857142857143,9.1984822336241347,131,4.2258064516129032
8,41,2024-03-31,62,16.42,185.93,46,39,52,17.1185714285714286,1.9703358719456594,1406,45.3548387096774194
9,42,2024-03-31,39,16.26,129.41,26,20,19,20.5985714285714286,3.3168829710555302,721,23.2580645161290323


In [15]:
# контроль: view не порожній, а NULL у lag-ознаках немає
run("""
SELECT
    COUNT(*)                                    AS n_rows,
    COUNT(DISTINCT pickup_zone_id)              AS n_zones,
    COUNT(*) FILTER (WHERE trips_lag_30d IS NULL) AS null_lag_30d
FROM v_t7_hw_taxi_features;
""")

,n_rows,n_zones,null_lag_30d
0,87,87,0


Що буде, якщо прибрати PARTITION BY pickup_zone_id: LAG на першому дні нової zone бере значення з останнього рядка попередньої zone. Нижче те саме порівняння на перших днях кожної zone: колонка lag_without_partition підтягує чужу історію, а lag_with_partition коректно дає NULL.

In [16]:
run("""
WITH compared AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        LAG(trip_count, 1) OVER (ORDER BY pickup_zone_id, ts_day)                      AS lag_without_partition,
        LAG(trip_count, 1) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day)          AS lag_with_partition,
        ROW_NUMBER()       OVER (PARTITION BY pickup_zone_id ORDER BY ts_day)          AS rn
    FROM t7_hw_taxi_daily
)
SELECT pickup_zone_id, ts_day, trip_count, lag_without_partition, lag_with_partition
FROM compared
WHERE rn = 1
ORDER BY pickup_zone_id
LIMIT 10;
""")

,pickup_zone_id,ts_day,trip_count,lag_without_partition,lag_with_partition
0,1,2024-03-13,1,NaN,None
1,2,2024-03-09,1,1.0,None
2,3,2024-03-01,1,1.0,None
3,4,2024-03-01,26,1.0,None
4,6,2024-03-03,1,57.0,None
5,7,2024-03-01,11,1.0,None
6,8,2024-03-22,1,17.0,None
7,9,2024-03-01,1,1.0,None
8,10,2024-03-01,7,1.0,None
9,11,2024-03-01,1,3.0,None


### Інтерпретація time-series частини

- **Гранулярність:** один рядок `t7_hw_taxi_daily` це `pickup_zone_id × ts_day`. День рахується за локальним нью-йоркським днем, тому `ts_day` має тип DATE, а момент поїздки в окремій typed-таблиці зберігається як TIMESTAMPTZ (raw-час інтерпретовано через `AT TIME ZONE 'America/New_York'`).
- **Очищення:** `fare_amount` у межах (0; 500) і `trip_distance` у межах (0; 100). Без цього від'ємні тарифи та аномальні дистанції спотворили б середні й волатильність.
- **Lag-ознаки:** `LAG(trip_count, n) OVER (PARTITION BY pickup_zone_id ORDER BY ts_day)`. Перші n днів кожної зони мають NULL, бо історії ще немає. Це чесна відсутність даних, а не помилка. Майбутнє не використовується, тому витоку даних (data leakage) немає.
- **Rolling 7d / 30d:** вікно `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` (і `'29 days'` для 30 днів) по `ts_day::TIMESTAMP`. Колонка `days_in_window` показує, що на початку періоду вікно неповне (1, 2, 3 ... днів). Для рядків, де `days_in_window` менше за розмір вікна, значення ненадійне, і при навчанні моделі їх краще відкидати або позначати. STDDEV_SAMP на одному рядку дає NULL.
- **ROWS чи RANGE:** RANGE з INTERVAL рахує саме календарні дні, навіть якщо в зоні пропущені дні. ROWS рахував би рядки і при пропусках захопив би довший період (off-by-one помилка). Для lag і expanding ознак застосовано ROWS/LAG за рядками, бо calendar-spine повний (кожна зона має рядок на кожен день).
- **Expanding:** `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` дає накопичену суму та середнє від початку періоду до поточного дня.
- **PARTITION BY:** без `PARTITION BY pickup_zone_id` LAG на першому дні нової зони бере значення останнього дня попередньої зони. Це тихий витік між сутностями, і демонстрація у ноутбуці це показує.

## Завдання 3. JSONB-таблиця для подій

In [17]:
run("""
DROP TABLE IF EXISTS t7_hw_events;

CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;
""")

In [18]:
run("""
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
""")

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


### JSONB Query 1. Containment @>

In [19]:
run("""
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;
""")

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 03:00:00+02:00,"{'total': 53.0, 'order_id': 'o_100003', 'items..."
1,18,19,2024-03-14 03:00:00+02:00,"{'total': 68.0, 'order_id': 'o_100018', 'items..."
2,33,34,2024-02-28 03:00:00+02:00,"{'total': 83.0, 'order_id': 'o_100033', 'items..."
3,48,49,2024-02-13 03:00:00+02:00,"{'total': 98.0, 'order_id': 'o_100048', 'items..."
4,63,64,2024-03-29 03:00:00+02:00,"{'total': 113.0, 'order_id': 'o_100063', 'item..."
5,78,79,2024-03-14 03:00:00+02:00,"{'total': 128.0, 'order_id': 'o_100078', 'item..."
6,93,94,2024-02-28 03:00:00+02:00,"{'total': 143.0, 'order_id': 'o_100093', 'item..."
7,108,109,2024-02-13 03:00:00+02:00,"{'total': 158.0, 'order_id': 'o_100108', 'item..."
8,123,124,2024-03-29 03:00:00+02:00,"{'total': 173.0, 'order_id': 'o_100123', 'item..."
9,138,139,2024-03-14 03:00:00+02:00,"{'total': 188.0, 'order_id': 'o_100138', 'item..."


### JSONB Query 2. Key existence ? та ?|

In [20]:
run("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;
""")

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quanti..."
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3..."
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity':..."
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quanti..."
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3..."
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity':..."
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quanti..."
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3..."
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity':..."
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quanti..."


In [21]:
run("""
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;
""")

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5'..."
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5'..."
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5'..."


Оператори ?, ?| і ?& перевіряють лише top-level ключі. Щоб індексувати такі запити через GIN, використовують default jsonb_ops, а не jsonb_path_ops.

### JSONB Query 3. JSONPath

In [22]:
run("""
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;
""")

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [23]:
run("""
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;
""")

,event_id,order_id,total
0,253,o_100253,303.00
1,258,o_100258,308.00
2,263,o_100263,313.00
3,268,o_100268,318.00
4,273,o_100273,323.00
5,278,o_100278,328.00
6,283,o_100283,333.00
7,288,o_100288,338.00
8,293,o_100293,343.00
9,298,o_100298,348.00


### Інтерпретація JSONB-запитів

- **Різні схеми payload:** 5 типів подій (`page_view`, `click`, `add_to_cart`, `checkout`, `recommendation_shown`) мають різні набори ключів в одній колонці. У цьому головна перевага JSONB: нові типи подій не потребують `ALTER TABLE`.
- **`@>` (containment):** перевіряє, що документ містить заданий фрагмент (наприклад `{"payment_method": "card"}`). Саме цей оператор підтримують GIN-індекси.
- **`?` і `?|`:** перевіряють наявність ключа лише на верхньому рівні. Вкладені ключі так не знайти.
- **JSONPath:** `jsonb_path_exists` фільтрує за умовою (`$.total > 300`), `jsonb_path_query` розгортає масив (`$.recommendations[*]`) у окремі рядки. Для аналітики це зручна заміна `jsonb_array_elements`.
- **Типи:** значення, які діставали через `->>`, мають тип text. Для порівняння чи агрегації їх треба приводити (`::numeric`), інакше буде лексикографічне порівняння.

## Завдання 4. GIN-index demo і EXPLAIN ANALYZE

### Step 1. Containment query без GIN-індексу

In [24]:
run("""
DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops;
DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops;
DROP INDEX IF EXISTS idx_t7_events_device_btree;

ANALYZE t7_hw_events;
""")

In [25]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
""")

Aggregate  (cost=1759.03..1759.04 rows=1 width=8) (actual time=43.777..43.777 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=3610 width=0) (actual time=0.027..43.519 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.233 ms
Execution Time: 43.805 ms


### Step 2. GIN-індекс для containment / JSONPath workload

In [26]:
run("""
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);

ANALYZE t7_hw_events;
""")

In [27]:
run("""
SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size,
       pg_size_pretty(pg_relation_size('t7_hw_events')) AS table_size;
""")

,gin_index_size,table_size
0,1144 kB,8000 kB


### Step 3. Той самий containment query після GIN

In [28]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
""")

Aggregate  (cost=1072.61..1072.62 rows=1 width=8) (actual time=4.900..4.901 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=26.62..1066.04 rows=2628 width=0) (actual time=0.473..4.724 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..25.97 rows=2628 width=0) (actual time=0.372..0.373 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=16
Planning Time: 1.656 ms
Execution Time: 4.940 ms


In [29]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
""")

Aggregate  (cost=1066.95..1066.96 rows=1 width=8) (actual time=4.079..4.080 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=25.32..1061.00 rows=2379 width=0) (actual time=0.425..3.923 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..24.72 rows=2379 width=0) (actual time=0.309..0.309 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.246 ms
Execution Time: 4.132 ms


### Step 4. Коли GIN не допомагає: path-specific text equality

Запит payload ->> 'device' = 'mobile' не є containment, тому GIN jsonb_path_ops його не прискорює. Спочатку план без expression index.

In [30]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
""")

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=8.682..8.683 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.020..8.326 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.083 ms
Execution Time: 8.706 ms


In [31]:
run("""
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));

ANALYZE t7_hw_events;
""")

In [32]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
""")

Aggregate  (cost=1243.56..1243.57 rows=1 width=8) (actual time=3.342..3.344 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=97.13..1222.64 rows=8367 width=0) (actual time=0.623..2.965 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..95.04 rows=8367 width=0) (actual time=0.498..0.498 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=19 read=1
Planning Time: 1.655 ms
Execution Time: 3.383 ms


### Інтерпретація EXPLAIN (ANALYZE, BUFFERS) для GIN

**Containment `payload @> {"payment_method": "card"}`.**

- До індексу: `Seq Scan` по всій таблиці (1000 буферів, Rows Removed by Filter ≈ 46 тис.), час виконання близько 50 мс, cost ≈ 1760.
- Після `GIN (payload jsonb_path_ops)`: `Bitmap Index Scan` на GIN-індексі, потім `Bitmap Heap Scan`; час близько 7 мс, cost ≈ 1114. Індекс сам читає лише кілька сторінок, а решту часу забирає читання рядків у heap.
- Вибірковість: запит повертає близько 7% рядків. Для рідкісніших значень виграш був би ще більшим. Якщо повертається більша частина таблиці, планувальник правильно обере Seq Scan.

**Чому GIN не допомагає для `payload ->> 'device' = 'mobile'`.** Оператор `->>` виймає текст, і запит це рівність виразу, а не containment. GIN з `jsonb_path_ops` або `jsonb_ops` не підтримує такий вираз, тому план залишається `Seq Scan`. Потрібен **expression B-tree index** на `(payload ->> 'device')`. Після нього план стає `Bitmap Index Scan` по `idx_t7_events_device_btree`, і час падає приблизно з 13 мс до 5 мс.

**jsonb_path_ops vs jsonb_ops.** `jsonb_path_ops` менший і швидший для `@>` та JSONPath, але не підтримує `?`, `?|`, `?&`. `jsonb_ops` універсальніший (підтримує ключові оператори), але більший і повільніший. Розмір GIN-індексу у цьому прогоні близько 1,1 МБ при таблиці близько 8 МБ.

**Ціна GIN.** Кожен INSERT і UPDATE оновлює індекс, тому write-навантаження та розмір на диску зростають.

## Завдання 5. JSONB vs normalized і Reflection

### Step 1. Нормалізована таблиця для checkout-подій

In [33]:
run("""
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
""")

In [34]:
run("""
SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;
""")

,n_checkouts
0,10000


### Step 2. Один запит двома способами

JSONB-версія:

In [35]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
""")

Limit  (cost=1953.98..1953.99 rows=3 width=72) (actual time=12.445..12.446 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1953.98..1967.81 rows=5531 width=72) (actual time=12.444..12.444 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1799.53..1882.49 rows=5531 width=72) (actual time=12.422..12.433 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1649.93 rows=9973 width=132) (actual time=0.012..7.199 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=14 read=2
Planning Time: 0.167 ms
Execution Time: 12.540 

Normalized-версія:

In [36]:
show_plan("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
""")

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=2.738..2.739 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=2.737..2.738 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=2.727..2.729 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.011..0.626 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=3
Planning Time: 0.145 ms
Execution Time: 2.766 ms


### Інтерпретація JSONB vs normalized

Один і той самий запит (середній `total` за способом оплати для checkout-подій) виконано двома способами:

| | JSONB `t7_hw_events` | Normalized `t7_hw_checkouts_norm` |
|---|---|---|
| Прочитано буферів | 1000 (вся таблиця подій) | 94 (лише checkout-рядки) |
| Час виконання | ≈ 12-17 мс | ≈ 3-4 мс |
| Типи колонок | `total` витягується з text і кастується | `total NUMERIC`, перевірка на рівні схеми |
| Валідація | немає, у JSONB може бути будь-що | NOT NULL, CHECK, типи |

Нормалізована таблиця швидша, бо містить лише потрібні рядки й колонки та має статистику планувальника по кожній колонці. JSONB зручніший, коли схема змінюється або різні події мають різні поля.

### Step 3. Bonus: MongoDB-equivalent (pseudocode, не виконується у PostgreSQL)

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode, не виконується у PostgreSQL

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

Різниця підходів. PostgreSQL JSONB зручний, коли потрібні SQL, JOIN, transactions і зв'язок із нормалізованими таблицями. MongoDB aggregation pipeline природний для document-first workloads. High-volume raw clickstream може краще жити в document store або data lake, а curated ML features зручніше тримати в typed SQL-таблицях.

## Reflection

**1. Як змінився план після GIN.** Для `payload @> {"payment_method": "card"}` до індексу був `Seq Scan` з 1000 буферами та ≈ 50 мс (cost ≈ 1760). Після `GIN (jsonb_path_ops)` з'явилися `Bitmap Index Scan` і `Bitmap Heap Scan`, час ≈ 7 мс (cost ≈ 1114). Тобто індекс замінив повне сканування вибіркою потрібних сторінок.

**2. Чому GIN не допоміг для `->> 'device'`.** Це рівність над текстовим виразом, а не containment, і GIN-оператори `@>`, `?`, `@?` її не обслуговують. Тому залишився `Seq Scan` (≈ 13 мс). Допоміг expression B-tree на `(payload ->> 'device')`: `Bitmap Index Scan`, ≈ 5 мс.

**3. Компроміс GIN.** Читання швидшає, але кожен INSERT/UPDATE оновлює індекс, і він займає місце (≈ 1,1 МБ на таблицю ≈ 8 МБ). Для write-heavy event-log з рідкими пошуками GIN може не окупитися. Для read-heavy аналітики він корисний. Тому перед створенням індексу варто оцінити співвідношення читань і записів та вибірковість типових запитів, а також порівняти `jsonb_path_ops` (менший, лише `@>` і JSONPath) із `jsonb_ops` (більший, але підтримує `?` і `?|`).

**4. JSONB чи normalized.** Для стабільних, часто запитуваних полів (як checkout: `order_id`, `total`, `payment_method`) краще normalized: менше буферів (94 проти 1000), типи та CHECK, статистика. JSONB доречний для гетерогенних подій і швидких змін схеми, де заздалегідь невідомо, які поля знадобляться. Практичний підхід: зберігати сирі події у JSONB, а ті поля, за якими постійно фільтрують і агрегують, виносити у нормалізовані колонки чи таблиці.

**5. Додавання `refund`.** У JSONB достатньо почати писати нові події з новим payload без міграції, але валідації немає. У normalized-моделі потрібна нова таблиця або `ALTER TABLE`, зате типи й обмеження гарантовано правильні. Якщо refund стане стабільним і важливим, його варто винести з JSONB у нормалізовану таблицю, а гнучкий JSONB залишити для решти подій.

## Підсумок

- Побудовано відтворюваний pipeline: Parquet (або stub) → `t7_hw_taxi_daily` → lag / rolling / expanding ознаки у view `v_t7_hw_taxi_features`.
- Event-log у JSONB з 5 типами подій, запити через `@>`, `?`, `?|`, JSONPath.
- GIN прискорює containment, expression B-tree потрібен для `->>`, а нормалізована таблиця виграє для стабільних полів.
- Усі таблиці й індекси мають префікс `t7_hw_` / `idx_t7_`, notebook можна безпечно запускати повторно (DROP IF EXISTS).